# Correlate cell entry in different cells

In [ ]:
# get variables from snakemake
entry_csv = snakemake.input.entry_csv
site_numbering_map_csv = snakemake.input.site_numbering_map
corr_chart = snakemake.output.corr_chart

# Imports
import itertools

import altair as alt

import pandas as pd

_ = alt.data_transformers.disable_max_rows()

In [ ]:
# Read data
entry_types = [
    "entry in a23 cells",
    "entry in a26 cells",
    "entry in mix of a23 and a26 cells",
]
entry = pd.read_csv(entry_csv)
assert set(entry_types).issubset(entry.columns)

site_numbering_map = pd.read_csv(site_numbering_map_csv)

entry = (
    entry
    [["site", "sequential_site", "wildtype", "mutant", *entry_types]]
    .merge(site_numbering_map[["sequential_site", "rbs_region"]], on="sequential_site", how="left", validate="m:1")
    .drop(columns="sequential_site")
    .assign(mutation=lambda x: x["wildtype"] + x["site"].astype(str) + x["mutant"])
)

entry_means = (
    entry
    .groupby(["site", "wildtype", "rbs_region"], as_index=False)
    .aggregate(**{col: pd.NamedAgg(col, "mean") for col in entry_types})
    .assign(site=lambda x: x["wildtype"] + x["site"].astype(str))
    .drop(columns="wildtype")
)

entry_means

In [ ]:
site_selection = alt.selection_point(
    fields=["site"], on="mouseover", empty=False
)

rbs_region_colors = {
    "RBS 130-loop": '#F0E442',
    "RBS 150-loop": '#E69F00',
    "RBS 190-loop": '#56B4E9',
    "RBS 220-loop": '#009E73',
    "RBS base": '#0072B2',
    "outside RBS": '#CCCCCC',
}

assert set(entry_means["rbs_region"]).issubset(rbs_region_colors)

base = (
    alt.Chart(entry_means)
    .add_params(site_selection)
    .encode(
        alt.Fill(
            "rbs_region",
            title="receptor-binding site (RBS) region",
            scale=alt.Scale(domain=rbs_region_colors.keys(), range=rbs_region_colors.values()),
        ),
        strokeWidth=alt.condition(site_selection, alt.value(2), alt.value(0.5)),
        size=alt.condition(site_selection, alt.value(90), alt.value(50)),
        tooltip=["site", alt.Tooltip("rbs_region", title="RBS region")],
    )
    .mark_circle(stroke="black", fillOpacity=0.7)
    .properties(width=250, height=250)
)

scatters = []
for entry_type1, entry_type2 in itertools.combinations(entry_types, 2):
    scatter = (
        base
        .encode(
            alt.X(entry_type1, scale=alt.Scale(domain=[-5, 1])),
            alt.Y(entry_type2, scale=alt.Scale(domain=[-5, 1])),
        )
    )
    scatters.append(scatter)

chart = (
    alt.hconcat(*scatters)
    .configure_axis(grid=False, titleFontSize=16, titleFontWeight="normal", labelFontSize=12)
    .configure_legend(
        orient="bottom",
        titleFontSize=16,
        labelFontSize=16,
        titleLimit=500,
    )
    .properties(
        title=alt.TitleParams(
            "Average effects of mutations at each site on entry in each cell type",
            anchor="middle",
            fontSize=16,
        ),
    )
)

chart.save(corr_chart)

chart